# Giai đoạn 14: Tải bộ dữ liệu ngoài ScaleOLDNEW lên Drive (để kiểm định độc lập)

**Nguồn:** Barborica và cộng sự (2023), *Front. Hum. Neurosci.* 17:1154038, giấy phép CC BY. Dữ liệu tại http://epi.fizica.unibuc.ro/scalesoldnew/

**Nội dung:** 12 bệnh nhân × 7 block. Mỗi block là bản ghi liên tục khoảng 5,6 phút, scalp EEG (khoảng 30–40 kênh) và SEEG trong cùng một file, 4096 Hz, khoảng 0,5–1,1 GB. Tổng cộng **khoảng 63 GB**.

**Notebook này làm gì:**
1. Liệt kê toàn bộ file trên server (`manifest.json`).
2. Đo tốc độ tải từ Colab, với 1 kết nối và với 16 kết nối song song.
3. Chép toàn bộ **file nhỏ** (header, marker, montage, bản EEGLAB, file Excel toạ độ) vào `MyDrive/iEEG_S2I/data/scalesoldnew/raw/`.
4. Với **từng block**:
   - tải `.dat` về ổ local của Colab bằng `aria2c` (16 kết nối song song, tiếp tục được khi đứt);
   - nếu bật `SAVE_RAW`, chép file thô lên Drive;
   - **tiền xử lý bằng đúng các bộ lọc của ds004752** (notch 50/100 Hz, band-pass 0,5–70 Hz zero-phase, hạ xuống 200 Hz), lưu vào `.../scalesoldnew/proc/SEEGxx_BlockK.npz` (khoảng 50 MB mỗi block);
   - xoá bản local.

**Tiếp tục được:** khi Colab ngắt, chỉ cần chạy lại cell 1, 2 và 5. Block đã xong sẽ được bỏ qua.

**Runtime: CPU là đủ.** Thời gian phụ thuộc gần như hoàn toàn vào tốc độ server (đo ở bước 4).

⚠️ **Dung lượng Drive:** `SAVE_RAW = True` cần khoảng **70 GB trống**. Nếu không đủ, đặt `SAVE_RAW = False`: chỉ lưu bản đã tiền xử lý (khoảng 5 GB), vẫn đủ cho mọi phân tích của bài.

In [ ]:
#@title 1. Mount Drive + công cụ
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, shutil
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
SC_ROOT  = f'{ROOT}/data/scalesoldnew'
SAVE_RAW = True                                #@param {type:"boolean"}
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
os.makedirs(SC_ROOT, exist_ok=True)
!apt-get -qq install -y aria2 > /dev/null
!pip -q install mne
du = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: còn trống {du.free / 1e9:.0f} GB / tổng {du.total / 1e9:.0f} GB')
print(f'Ổ local Colab: còn trống {shutil.disk_usage("/content").free / 1e9:.0f} GB')
if SAVE_RAW and du.free < 75e9:
    print('⚠️  Drive trống dưới 75 GB: nên đặt SAVE_RAW = False (chỉ lưu bản đã tiền xử lý, khoảng 5 GB)')

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/data/scales_fetch.py
"""
Download and preprocess the ScaleOLDNEW dataset (Barborica et al. 2023, Front. Hum. Neurosci. 17:1154038;
http://epi.fizica.unibuc.ro/scalesoldnew/) for external validation.

12 patients x 7 blocks of continuous raw recordings in AnyWave ADES format: <name>.ades (text header: sampling
rate, number of samples, one "channel = type" line per channel) + <name>.dat (float32, sample-major / multiplexed,
microvolts, ~1.1 GB per block at 4096 Hz). Scalp EEG and SEEG are in the same file.

Commands
  crawl      list every file of the repository (recursive) -> manifest.json
  speedtest  download the first N MB of one block with 1 and with 16 connections, print MB/s
  fetch      for every file: small files are copied to the Drive mirror; every block .dat is downloaded to local
             disk with aria2c (parallel range requests, resumable), optionally copied to Drive (--save_raw), and
             preprocessed with exactly the ds004752 filters (notch 50/100 Hz, band-pass 0.5-70 Hz zero-phase FIR,
             resampling to 200 Hz) into <proc>/<patient>_Block<k>.npz; the local copy is then deleted.
             Resumable: finished blocks are skipped.

Usage
-----
python -m s2i.data.scales_fetch crawl     --out DRIVE_ROOT
python -m s2i.data.scales_fetch speedtest --out DRIVE_ROOT
python -m s2i.data.scales_fetch fetch     --out DRIVE_ROOT [--save_raw] [--patients SEEG89 ...] [--blocks 1 2 3]
"""
import argparse
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request

import numpy as np

BASE = "http://epi.fizica.unibuc.ro"
ROOT_PATH = "/scalesoldnew/"
FS = 200
NOTCH = (50.0, 100.0)
BAND = (0.5, 70.0)
CONTACT = re.compile(r"^[A-Z]{1,2}'?\d{1,2}$")       # SEEG contact names, e.g. A01, Q'03 (not OSAT, PR, Pleth)
SMALL = 60e6                                          # files below this size go straight to the Drive mirror


def log(msg):
    print(f"{time.strftime('%H:%M:%S')} | {msg}", flush=True)


def get(url, timeout=120, tries=6):
    import urllib.error
    for k in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                return r.read()
        except urllib.error.HTTPError as e:           # 403/404: the server refuses this file type, do not retry
            if e.code in (403, 404):
                raise
            if k == tries - 1:
                raise
            log(f"  retry {k + 1} (HTTP {e.code}) {url}")
            time.sleep(5 * (k + 1))
        except Exception as e:                        # flaky server: back off and retry
            if k == tries - 1:
                raise
            log(f"  retry {k + 1} ({type(e).__name__}) {url}")
            time.sleep(5 * (k + 1))


def listing(path):
    html = get(BASE + path).decode("latin-1")
    out = []
    for m in re.finditer(r"(\d+|&lt;dir&gt;)\s+<A HREF=\"([^\"]+)\">", html):
        size, href = m.group(1), m.group(2)
        if href.rstrip("/") == path.rstrip("/").rsplit("/", 1)[0] or not href.startswith(path):
            continue
        out.append((href, None if size == "&lt;dir&gt;" else int(size)))
    return out


def crawl(a):
    """Breadth-first, one level at a time, 16 directory listings in parallel (the server answers slowly)."""
    from concurrent.futures import ThreadPoolExecutor
    files, level = [], [ROOT_PATH]
    with ThreadPoolExecutor(16) as ex:
        while level:
            nxt = []
            for p, items in zip(level, ex.map(listing, level)):
                for href, size in items:
                    if size is None:
                        nxt.append(href)
                    else:
                        files.append({"path": href, "size": size})
            log(f"listed {len(level)} folders ({len(files)} files so far)")
            level = nxt
    files.sort(key=lambda f: f["path"])
    os.makedirs(a.out, exist_ok=True)
    json.dump(files, open(os.path.join(a.out, "manifest.json"), "w"), indent=0)
    tot = sum(f["size"] for f in files)
    big = [f for f in files if f["size"] >= SMALL]
    log(f"{len(files)} files, {tot / 1e9:.1f} GB in total; {len(big)} large files ({sum(f['size'] for f in big) / 1e9:.1f} GB)")


def aria2(url, dest_dir, name, conns=16):
    cmd = ["aria2c", f"-x{conns}", f"-s{conns}", "-k4M", "-c", "--file-allocation=none", "--max-tries=20",
           "--retry-wait=10", "--timeout=120", "--console-log-level=warn", "--summary-interval=60",
           "-d", dest_dir, "-o", name, url]
    subprocess.run(cmd, check=True)


def speedtest(a):
    files = json.load(open(os.path.join(a.out, "manifest.json")))
    dat = next(f for f in files if f["path"].endswith(".dat"))
    url = BASE + dat["path"]
    from concurrent.futures import ThreadPoolExecutor
    n = int(a.mb * 1e6)

    def part(rng):
        req = urllib.request.Request(url, headers={"Range": f"bytes={rng[0]}-{rng[1]}"})
        with urllib.request.urlopen(req, timeout=600) as r:
            return len(r.read())
    for conns in (1, 16):
        edges = np.linspace(0, n, conns + 1).astype(int)
        t0 = time.time()
        with ThreadPoolExecutor(conns) as ex:
            got = sum(ex.map(part, [(edges[k], edges[k + 1] - 1) for k in range(conns)]))
        dt = time.time() - t0
        log(f"{conns:2d} connection(s): {got / 1e6:.1f} MB in {dt:.0f} s = {got / 1e6 / max(dt, 1e-9):.2f} MB/s")
    tot = sum(f["size"] for f in files) / 1e9
    log(f"repository size {tot:.1f} GB; at 1 MB/s that is {tot * 1e3 / 3600:.0f} h")


def read_header(path):
    fs, n, chans = None, None, []
    for ln in open(path, encoding="latin-1").read().splitlines():
        if ln.startswith("#") or "=" not in ln:
            continue
        k, v = [x.strip() for x in ln.split("=", 1)]
        if k == "samplingRate":
            fs = float(v)
        elif k == "numberOfSamples":
            n = int(v)
        else:
            chans.append((k, v))
    return fs, n, chans


def read_markers(path):
    out = []
    if not os.path.exists(path):
        return out
    for ln in open(path, encoding="latin-1").read().splitlines():
        p = [x.strip() for x in ln.split("\t")]
        if len(p) >= 3 and not ln.startswith("//"):
            try:
                out.append((p[0], float(p[2])))
            except ValueError:
                pass
    return out


def preprocess_block(dat, ades, mrk, bad):
    import mne
    from s2i.data.preprocess_ds004752 import filt_resample
    mne.set_log_level("ERROR")
    fs, n, chans = read_header(ades)
    C = len(chans)
    size = os.path.getsize(dat)
    assert size == 4 * C * n, f"size {size} != 4 x {C} channels x {n} samples"
    X = np.memmap(dat, dtype="<f4", mode="r", shape=(n, C))
    groups = {"eeg": [i for i, (c, t) in enumerate(chans) if t == "EEG"],
              "seeg": [i for i, (c, t) in enumerate(chans) if t == "SEEG" and CONTACT.match(c)],
              "aux": [i for i, (c, t) in enumerate(chans) if t in ("ECG", "EMG")]}
    out = {"fs": FS, "fs_raw": fs, "n_raw": n, "bad": bad, "markers": read_markers(mrk)}
    for g, idx in groups.items():
        parts = []
        for k in range(0, len(idx), 24):                       # 24 channels at a time (memory)
            sub = idx[k:k + 24]
            x = np.ascontiguousarray(X[:, sub].T, dtype=np.float64) * 1e-6       # microvolts -> volts
            raw = mne.io.RawArray(x, mne.create_info([chans[i][0] for i in sub], fs, "misc"), verbose=False)
            parts.append(filt_resample(raw).astype(np.float32) * 1e6)          # back to microvolts
        out[g] = np.concatenate(parts, 0) if parts else np.zeros((0, 0), np.float32)
        out[g + "_names"] = [chans[i][0] for i in idx]
    del X
    return out


def fetch(a):
    files = json.load(open(os.path.join(a.out, "manifest.json")))
    raw_root, proc_root = os.path.join(a.out, "raw"), os.path.join(a.out, "proc")
    tmp = a.tmp
    os.makedirs(proc_root, exist_ok=True)
    os.makedirs(tmp, exist_ok=True)
    keep = lambda p: (not a.patients or any(f"/{q}/" in p for q in a.patients))      # noqa: E731
    small = [f for f in files if f["size"] < SMALL and keep(f["path"])]
    big = [f for f in files if f["size"] >= SMALL and keep(f["path"])]
    # 1) small files -> Drive mirror (headers, markers, montages, EEGLAB .set/.fdt, spreadsheets)
    from concurrent.futures import ThreadPoolExecutor

    def mirror(f):
        dst = os.path.join(raw_root, f["path"][len(ROOT_PATH):])
        if os.path.exists(dst) and os.path.getsize(dst) == f["size"]:
            return "ok"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        try:
            data = get(BASE + f["path"], timeout=300)
        except Exception as e:                         # e.g. 404 for file types the server does not serve (.sel)
            return f"skipped ({type(e).__name__}: {getattr(e, 'code', '')})"
        with open(dst + ".part", "wb") as fh:
            fh.write(data)
        os.replace(dst + ".part", dst)
        return "new"
    with ThreadPoolExecutor(16) as ex:                 # the server answers slowly; fetch in parallel
        status = list(ex.map(mirror, small))
    skipped = [(f["path"], s) for f, s in zip(small, status) if s.startswith("skipped")]
    log(f"small files: {len(small)} listed, {status.count('new')} new, {len(skipped)} skipped")
    for ext in sorted({os.path.splitext(p)[1] for p, _ in skipped}):
        n = sum(p.endswith(ext) for p, _ in skipped)
        log(f"  skipped {n} '{ext}' file(s), e.g. {next(p for p, _ in skipped if p.endswith(ext))}")
    needed = [p for p, _ in skipped if p.endswith((".ades", ".ades.bad", ".ades.mrk", ".mrk"))]
    assert not needed, f"files needed for preprocessing could not be fetched: {needed[:5]}"
    # 2) block .dat files (and EEGLAB .fdt above SMALL) -> local -> preprocess -> Drive
    for f in big:
        rel = f["path"][len(ROOT_PATH):]
        name = os.path.basename(rel)
        dst_raw = os.path.join(raw_root, rel)
        is_block = name.endswith(".dat")
        if is_block:
            m = re.search(r"(SEEG\d+)Block(\d+)", name)
            if a.blocks and int(m.group(2)) not in a.blocks:
                continue
            proc = os.path.join(proc_root, f"{m.group(1)}_Block{m.group(2)}.npz")
            done_raw = (not a.save_raw) or (os.path.exists(dst_raw) and os.path.getsize(dst_raw) == f["size"])
            if os.path.exists(proc) and done_raw:
                continue
        elif os.path.exists(dst_raw) and os.path.getsize(dst_raw) == f["size"]:
            continue
        t0 = time.time()
        local = os.path.join(tmp, name)
        if os.path.exists(dst_raw) and os.path.getsize(dst_raw) == f["size"]:
            shutil.copy(dst_raw, local)                                         # raw already on Drive
        else:
            free = shutil.disk_usage(tmp).free
            assert free > f["size"] * 1.2, f"not enough local disk ({free / 1e9:.1f} GB free)"
            aria2(BASE + f["path"], tmp, name)
        assert os.path.getsize(local) == f["size"], f"incomplete download {name}"
        dt = time.time() - t0
        log(f"{rel}: {f['size'] / 1e9:.2f} GB in {dt / 60:.1f} min ({f['size'] / 1e6 / max(dt, 1):.2f} MB/s)")
        if a.save_raw or not is_block:
            os.makedirs(os.path.dirname(dst_raw), exist_ok=True)
            shutil.copy(local, dst_raw + ".part")
            os.replace(dst_raw + ".part", dst_raw)
        if is_block:
            base = os.path.join(raw_root, os.path.dirname(rel), name[:-4])
            bad_f = base + ".ades.bad"
            bad = open(bad_f, encoding="latin-1").read().split() if os.path.exists(bad_f) else []
            mrk = base + ".ades.mrk" if os.path.exists(base + ".ades.mrk") else base + ".mrk"
            t1 = time.time()
            out = preprocess_block(local, base + ".ades", mrk, bad)
            np.savez(proc + ".part.npz", **{k: (np.array(v, dtype=object) if isinstance(v, list) else v)
                                            for k, v in out.items()})
            os.replace(proc + ".part.npz", proc)
            log(f"  preprocessed in {(time.time() - t1) / 60:.1f} min: {out['eeg'].shape[0]} scalp, "
                f"{out['seeg'].shape[0]} SEEG, {out['eeg'].shape[1] / FS / 60:.1f} min, {len(out['markers'])} markers")
        os.remove(local)
    log("fetch finished")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("crawl", "speedtest", "fetch"))
    ap.add_argument("--out", required=True, help="Drive root for this dataset (manifest, raw/, proc/)")
    ap.add_argument("--tmp", default="/content/scales_tmp")
    ap.add_argument("--save_raw", action="store_true", help="also keep the raw .dat files on Drive (~63 GB)")
    ap.add_argument("--patients", nargs="*", default=None)
    ap.add_argument("--blocks", nargs="*", type=int, default=None)
    ap.add_argument("--mb", type=float, default=64)
    ap.add_argument("--secs", type=int, default=300)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    {"crawl": crawl, "speedtest": speedtest, "fetch": fetch}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
#@title 2b. Kiểm tra (cả hai số phải > 0)
!grep -c "def preprocess_block" {CODE_DIR}/s2i/data/scales_fetch.py
!grep -c "def filt_resample" {CODE_DIR}/s2i/data/preprocess_ds004752.py

## 3. Liệt kê file trên server (một lần, vài phút)

In [ ]:
#@title 3. Crawl
if not os.path.exists(f'{SC_ROOT}/manifest.json'):
    !python -m s2i.data.scales_fetch crawl --out {SC_ROOT}
import json
files = json.load(open(f'{SC_ROOT}/manifest.json'))
print(len(files), 'file,', round(sum(f['size'] for f in files) / 1e9, 1), 'GB;',
      len([f for f in files if f['path'].endswith('.dat')]), 'block .dat')

## 4. Đo tốc độ tải (64 MB đầu của một block)

In [ ]:
#@title 4. Speed test
!python -m s2i.data.scales_fetch speedtest --out {SC_ROOT} --mb 64

## 5. Tải + tiền xử lý toàn bộ (chạy lại cell này sau mỗi lần Colab ngắt)
Thử trước với một bệnh nhân bằng cách thêm `--patients SEEG89`, kiểm tra ở cell 6, rồi mới chạy toàn bộ.

In [ ]:
#@title 5. Fetch
RAW_FLAG = '--save_raw' if SAVE_RAW else ''
!python -m s2i.data.scales_fetch fetch --out {SC_ROOT} {RAW_FLAG}

## 6. Tiến độ và kiểm tra một block

In [ ]:
#@title 6. Kiểm tra
import numpy as np
proc = sorted(glob.glob(f'{SC_ROOT}/proc/*.npz'))
print(f'{len(proc)} / 84 block đã tiền xử lý')
raws = glob.glob(f'{SC_ROOT}/raw/**/*.dat', recursive=True)
print(f'{len(raws)} file thô trên Drive ({sum(os.path.getsize(r) for r in raws) / 1e9:.1f} GB)')
if proc:
    z = np.load(proc[0], allow_pickle=True)
    print(os.path.basename(proc[0]), '| scalp', z['eeg'].shape, '| SEEG', z['seeg'].shape, '| fs', z['fs'],
          '| marker', len(z['markers']), '| kênh hỏng', list(z['bad'])[:10])
    print('scalp:', list(z['eeg_names'])[:45])
    print('SEEG :', list(z['seeg_names'])[:20], '...')
    print('biên độ scalp (µV, độ lệch chuẩn):', np.round(z['eeg'].std(1)[:8], 1))

## Sau khi tải xong
Gửi lại output của cell 3, 4 và 6.

Bước tiếp theo (Phase 15) sẽ **khoá trước protocol kiểm định ngoài** rồi mới chạy:
- encoder chỉ train trên ds004752;
- công thức của mô hình cuối;
- calib = 20% thời gian đầu, cách 1 cửa sổ;
- chuẩn hoá strict;
- tách hai nhóm amplifier (bệnh nhân 1–8 ghi bằng hai amplifier riêng, 9–12 bằng một amplifier chung) cho phân tích thành phần chung.